In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()
# GITHUB_PERSONAL_ACCESS_TOKEN is loaded from .env file


In [ ]:
!gh --version

!gh auth status

!gh auth login

In [ ]:
!gh repo create playlist-manager --public --description "Spotify-style playlist manager"
!git clone https://github.com/YOUR_GITHUB_USERNAME/playlist-manager.git
%cd playlist-manager

In [ ]:
readme = """# Playlist Manager

A simple Spotify-style playlist management feature.

## Features

- Create playlists
- Add songs
- Remove songs
- Search songs
- Like songs
- Recommend songs based on listening history

## Future AI Feature

An AI agent could analyze a user's listening history and automatically recommend songs that match their preferences.
"""

with open("README.md", "w", encoding="utf-8") as file:
    file.write(readme)

print("README.md created.")

In [ ]:
!git add README.md
!git commit -m "Add playlist manager README"
!git push origin main

In [ ]:
%%writefile github_webhook.py

from flask import Flask, request, jsonify

app = Flask(__name__)


@app.route("/webhook", methods=["POST"])
def webhook():

    event = request.headers.get("X-GitHub-Event")

    if event == "issues":
        data = request.get_json()

        action = data.get("action")
        issue = data.get("issue", {})

        if action == "opened":
            print("\nNew GitHub Issue Created")
            print("Title:", issue.get("title"))
            print("Author:", issue.get("user", {}).get("login"))

    return jsonify({"status": "received"})


@app.route("/")
def home():
    return "GitHub Webhook Server Running"


if __name__ == "__main__":
    app.run(
        host="127.0.0.1",
        port=5002,
        debug=False
    )

In [ ]:
import requests

payload = {
    "action": "opened",
    "issue": {
        "title": "Add playlist shuffle feature",
        "user": {
            "login": "test-user"
        }
    }
}

response = requests.post(
    "http://127.0.0.1:5002/webhook",
    headers={
        "X-GitHub-Event": "issues"
    },
    json=payload
)

print(response.json())

In [ ]:
## GitHub Webhook Configuration

Open the `playlist-manager` repository on GitHub.

Go to:

Settings
→ Webhooks
→ Add webhook

Payload URL:
http://YOUR_PUBLIC_WEBHOOK_URL/webhook

Content type:
application/json

Events:
Select "Issues"

Active:
Yes

Then click "Add webhook".

Create a test issue in the repository.

The webhook receiver will receive the `issues` event and print:

New GitHub Issue Created
Title: <issue title>
Author: <issue author>

In [ ]:
%%writefile github_webhook.py

from flask import Flask, request, jsonify

app = Flask(__name__)


@app.route("/webhook", methods=["POST"])
def webhook():

    event = request.headers.get("X-GitHub-Event")
    data = request.get_json() or {}

    if event == "pull_request":

        action = data.get("action")
        pull_request = data.get("pull_request", {})

        labels = [
            label.get("name", "").lower()
            for label in pull_request.get("labels", [])
        ]

        if action in ["opened", "synchronize", "reopened"] and "bug" in labels:

            title = pull_request.get("title")
            author = pull_request.get("user", {}).get("login")

            print("\nBug Pull Request Detected")
            print("Title:", title)
            print("Author:", author)

    return jsonify({"status": "received"})


@app.route("/")
def home():
    return "GitHub PR Webhook Server Running"


if __name__ == "__main__":
    app.run(
        host="127.0.0.1",
        port=5002,
        debug=False
    )

In [ ]:
import requests

payload = {
    "action": "opened",
    "pull_request": {
        "title": "Fix playlist loading bug",
        "user": {
            "login": "developer123"
        },
        "labels": [
            {"name": "bug"}
        ]
    }
}

response = requests.post(
    "http://127.0.0.1:5002/webhook",
    headers={
        "X-GitHub-Event": "pull_request"
    },
    json=payload
)

print(response.json())

In [ ]:
import requests

payload = {
    "action": "opened",
    "pull_request": {
        "title": "Fix playlist loading bug",
        "user": {
            "login": "developer123"
        },
        "labels": [
            {"name": "bug"}
        ]
    }
}

response = requests.post(
    "http://127.0.0.1:5002/webhook",
    headers={
        "X-GitHub-Event": "pull_request"
    },
    json=payload
)

print(response.json())

In [ ]:
## AI-Generated Code

Prompt given to ChatGPT:

"Generate a Python Flask webhook handler for GitHub that detects a newly created issue and posts a thank-you comment on that issue using the GitHub API."

Generated approach:

```python
import requests

def thank_issue(owner, repo, issue_number, token):
    url = f"https://api.github.com/repos/{owner}/{repo}/issues/{issue_number}/comments"

    headers = {
        "Authorization": f"Bearer {token}",
        "Accept": "application/vnd.github+json"
    }

    data = {
        "body": "Thank you for creating this issue!"
    }

    response = requests.post(
        url,
        headers=headers,
        json=data
    )

    return response.status_code

In [ ]:

---

# Cell 12 — Improved Working Version

```python
import os
import requests

GITHUB_TOKEN = os.getenv("GITHUB_TOKEN")


def thank_issue(owner, repo, issue_number):
    if not GITHUB_TOKEN:
        return "GITHUB_TOKEN is not configured"

    url = (
        f"https://api.github.com/repos/"
        f"{owner}/{repo}/issues/{issue_number}/comments"
    )

    headers = {
        "Authorization": f"Bearer {GITHUB_TOKEN}",
        "Accept": "application/vnd.github+json"
    }

    data = {
        "body": "Thank you for creating this issue! We will review it soon."
    }

    response = requests.post(
        url,
        headers=headers,
        json=data,
        timeout=10
    )

    if response.status_code == 201:
        return "Thank-you comment posted successfully"

    return f"GitHub API error: {response.status_code}"


print(
    thank_issue(
        "YOUR_GITHUB_USERNAME",
        "playlist-manager",
        1
    )
)

In [ ]:
%%writefile github_webhook.py

from flask import Flask, request, jsonify
import requests
import os

app = Flask(__name__)

GITHUB_TOKEN = os.getenv("GITHUB_TOKEN")


def thank_issue(owner, repo, issue_number):

    url = (
        f"https://api.github.com/repos/"
        f"{owner}/{repo}/issues/{issue_number}/comments"
    )

    headers = {
        "Authorization": f"Bearer {GITHUB_TOKEN}",
        "Accept": "application/vnd.github+json"
    }

    data = {
        "body": "Thank you for creating this issue! We will review it soon."
    }

    response = requests.post(
        url,
        headers=headers,
        json=data,
        timeout=10
    )

    return response.status_code == 201


@app.route("/webhook", methods=["POST"])
def webhook():

    event = request.headers.get("X-GitHub-Event")
    data = request.get_json() or {}

    if event == "issues" and data.get("action") == "opened":

        repository = data["repository"]
        issue = data["issue"]

        owner = repository["owner"]["login"]
        repo = repository["name"]
        issue_number = issue["number"]

        success = thank_issue(
            owner,
            repo,
            issue_number
        )

        if success:
            print("Thank-you comment posted.")
        else:
            print("Failed to post comment.")

    return jsonify({"status": "received"})


if __name__ == "__main__":
    app.run(
        host="127.0.0.1",
        port=5002,
        debug=False
    )